# **Cloud-Native GRIB2: High-Performance Access with grib2io and VirtualiZarr**

This notebook demonstrates how to efficiently access GRIB2 data stored on AWS S3 without streaming the entire file. We'll leverage standard `.idx` sidecar files to build a virtual Xarray dataset instantly.

## Notebook Convention

Use the standard grib2io xarray backend interface directly.
Pass grib2io-specific options as keyword arguments to `xr.open_dataset(...)` or `xr.open_mfdataset(...)` (for example `use_icechunk`, `storage_options`, `filters`, `max_workers`, `network_timeout`, and `max_concurrent_requests`).

## **The GRIB2 Challenge**

GRIB2 is a streaming format without a central metadata header. To create a virtual Zarr manifest, a standard scanner must read the file from start to finish to find message boundaries. For a multi-gigabyte file on S3, this normally means streaming all that data just to find the offsets.

### **The Solution: Standard sidecar Indices**
Public datasets (like NOAA GFS) often provide text-based `.idx` files alongside the GRIB files. `grib2io` can now parse these `.idx` files to extract byte offsets directly. This allows it to jump to each message and read only the small metadata headers (Sections 0-5), completely bypassing the large data payloads (Section 7) during the scanning phase.

In [ ]:
import xarray as xr
import grib2io
import fsspec
from pathlib import Path
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import KerchunkJSONParser
from obstore.store import LocalStore, S3Store
from obspec_utils.registry import ObjectStoreRegistry
import s3fs
import json
import os

## **1. Locating GFS Data on S3**

We'll use a public GFS dataset from the NOAA Open Data Program on AWS. Note the presence of the `.idx` file.

In [2]:
# Example S3 path for a GFS file
s3_bucket = "noaa-gfs-bdp-pds"
s3_path = "gfs.20240501/00/atmos/gfs.t00z.pgrb2.0p25.f000"
s3_url = f"s3://{s3_bucket}/{s3_path}"
# Standard sidecar index: s3://noaa-gfs-bdp-pds/gfs.20240501/00/atmos/gfs.t00z.pgrb2.0p25.f000.idx

# Verify the key exists in the bucket before trying to build references.
fs = s3fs.S3FileSystem(anon=True)
if not fs.exists(f"{s3_bucket}/{s3_path}"):
    raise FileNotFoundError(f"S3 object not found: {s3_url}")

## **2. Building the Virtual Manifest via Index**

By leveraging the `.idx` file, `grib2io` maps out the chunks instantly. The `ReferenceGenerator` uses this to create a Kerchunk manifest.

In [3]:
from grib2io.kerchunk import ReferenceGenerator

# Stream directly from S3 (no local download).
gen = ReferenceGenerator(
    s3_url,
    storage_options={
        "anon": True,
        "default_fill_cache": False,
        "default_cache_type": "none",
        "default_block_size": 131072,
    },
)

# This step is now extremely fast as it avoids streaming data payloads.
manifest = gen.generate()

manifest_path = "gfs_s3_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f)

## **3. Ingesting into VirtualiZarr**

Load the S3-backed manifest into VirtualiZarr for metadata manipulation.

In [4]:
manifest_file = Path(manifest_path).resolve()
manifest_url = manifest_file.as_uri()

registry = ObjectStoreRegistry(
    {
        manifest_url: LocalStore(prefix=str(manifest_file.parent)),
        "s3://noaa-gfs-bdp-pds": S3Store(bucket="noaa-gfs-bdp-pds", skip_signature=True),
    }
)

vds = open_virtual_dataset(
    url=manifest_url,
    registry=registry,
    parser=KerchunkJSONParser(),
    loadable_variables=[],
)
display(vds)

<xarray.Dataset> Size: 3GB
Dimensions:                              (valid_time: 1, mean_sea_level: 1,
                                          y: 721, x: 1440, hybrid_level: 1,
                                          hybrid_level_2: 2,
                                          entire_atmosphere: 1, surface: 1,
                                          planetary_boundary_layer: 1,
                                          isobaric_surface: 41,
                                          ...
                                          highest_tropospheric_freezing_level: 1,
                                          pressure_difference_from_ground: 1,
                                          pressure_difference_from_ground_2: 3,
                                          sigma_level: 4, sigma_level_2: 1,
                                          pressure_difference_from_ground_3: 1,
                                          potential_vorticity_surface: 2)
Coordinates: (12/35)
    valid_time                           (valid_time) int64 8B ManifestArray<...
    mean_sea_level                       (mean_sea_level) float64 8B Manifest...
    hybrid_level                         (hybrid_level) float64 8B ManifestAr...
    hybrid_level_2                       (hybrid_level_2) float64 16B Manifes...
    entire_atmosphere                    (entire_atmosphere) float64 8B Manif...
    surface                              (surface) float64 8B ManifestArray<s...
    ...                                   ...
    pressure_difference_from_ground      (pressure_difference_from_ground) float64 8B ManifestArray<shape=(1,), dtype=float64, chu...
    pressure_difference_from_ground_2    (pressure_difference_from_ground_2) float64 24B ManifestArray<shape=(3,), dtype=float64, ...
    sigma_level                          (sigma_level) float64 32B ManifestAr...
    sigma_level_2                        (sigma_level_2) float64 8B ManifestA...
    pressure_difference_from_ground_3    (pressure_difference_from_ground_3) float64 8B ManifestArray<shape=(1,), dtype=float64, c...
    potential_vorticity_surface          (potential_vorticity_surface) float64 16B ManifestArray<shape=(2,), dtype=float64, chunks...
Dimensions without coordinates: y, x
Data variables: (12/125)
    PRMSL                                (valid_time, mean_sea_level, y, x) float32 4MB ManifestArray<shape=(1, 1, 721, 1440), dtype=float32, chunks=(1, 1, ...
    CLMR                                 (valid_time, hybrid_level, y, x) float32 4MB ManifestArray<shape=(1, 1, 721, 1440), dtype=float32, chunks=(1, 1, 72...
    ICMR                                 (valid_time, hybrid_level, y, x) float32 4MB ManifestArray<shape=(1, 1, 721, 1440), dtype=float32, chunks=(1, 1, 72...
    RWMR                                 (valid_time, hybrid_level, y, x) float32 4MB ManifestArray<shape=(1, 1, 721, 1440), dtype=float32, chunks=(1, 1, 72...
    SNMR                                 (valid_time, hybrid_level, y, x) float32 4MB ManifestArray<shape=(1, 1, 721, 1440), dtype=float32, chunks=(1, 1, 72...
    GRLE                                 (valid_time, hybrid_level, y, x) float32 4MB ManifestArray<shape=(1, 1, 721, 1440), dtype=float32, chunks=(1, 1, 72...
    ...                                   ...
    UGRD_8                               (valid_time, potential_vorticity_surface, y, x) float32 8MB ManifestArray<shape=(1, 2, 721, 1440), dtype=float32, c...
    VGRD_8                               (valid_time, potential_vorticity_surface, y, x) float32 8MB ManifestArray<shape=(1, 2, 721, 1440), dtype=float32, c...
    TMP_8                                (valid_time, potential_vorticity_surface, y, x) float32 8MB ManifestArray<shape=(1, 2, 721, 1440), dtype=float32, c...
    HGT_7                                (valid_time, potential_vorticity_surface, y, x) float32 8MB ManifestArray<shape=(1, 2, 721, 1440), dtype=float32, c...
    PRES_4                               (valid_time, potential_vorticity_

## **4. Targeted Data Access**

Use the standard grib2io/xarray interface for dataset opening:

| Interface | Input | Use when… |
|---|---|---|
| `xr.open_dataset(url, engine="grib2io", use_icechunk=True, ...)` | GRIB2 file path / URI | You want a single call from file → Dataset |
| `xr.open_mfdataset(urls, engine="grib2io", use_icechunk=True, ...)` | list of GRIB2 paths / URIs | You want a combined multi-file Dataset |
| `xr.open_dataset(manifest_path, engine="grib2io")` | pre-built manifest file | You already have a saved manifest |

Notebook convention: pass grib2io-specific options directly as keyword arguments
to `xr.open_dataset(...)` / `xr.open_mfdataset(...)`.

All three paths route through the same backend UI and handle local/remote storage transparently.


In [ ]:
# ── Option A: one call from file path -> Dataset via grib2io backend ─────────
ds = xr.open_dataset(
    s3_url,
    engine="grib2io",
    use_icechunk=True,
    storage_options={"anon": True},
)

# ── Option B: if you already built (or loaded) a manifest file ───────────────
# ds = xr.open_dataset(manifest_path, engine="grib2io")

display(ds)

  2026-05-21T16:18:03.127812Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:196



<xarray.Dataset> Size: 3GB
Dimensions:                              (valid_time: 1, surface: 1, y: 721,
                                          x: 1440,
                                          pressure_difference_from_ground_2: 3,
                                          hybrid_level: 1,
                                          height_above_ground_4: 1,
                                          isobaric_surface: 41,
                                          ...
                                          height_above_ground_3: 2,
                                          height_above_ground_2: 3,
                                          altitude_above_mean_sea_level_2: 3,
                                          planetary_boundary_layer: 1,
                                          height_above_ground_7: 1,
                                          height_above_ground_5: 7)
Coordinates: (12/35)
  * valid_time                           (valid_time) datetime64[ns] 8B 2024-...
  * surface                              (surface) float64 8B 0.0
  * pressure_difference_from_ground_2    (pressure_difference_from_ground_2) float64 24B ...
  * hybrid_level                         (hybrid_level) float64 8B 1.0
  * height_above_ground_4                (height_above_ground_4) float64 8B 2.0
  * isobaric_surface                     (isobaric_surface) float64 328B 1.0 ...
    ...                                   ...
  * height_above_ground_3                (height_above_ground_3) float64 16B ...
  * height_above_ground_2                (height_above_ground_2) float64 24B ...
  * altitude_above_mean_sea_level_2      (altitude_above_mean_sea_level_2) float64 24B ...
  * planetary_boundary_layer             (planetary_boundary_layer) float64 8B ...
  * height_above_ground_7                (height_above_ground_7) float64 8B 6...
  * height_above_ground_5                (height_above_ground_5) float64 56B ...
Dimensions without coordinates: y, x
Data variables: (12/125)
    4LFTX                                (valid_time, surface, y, x) float32 4MB dask.array<chunksize=(1, 1, 721, 1440), meta=np.ndarray>
    CIN                                  (valid_time, surface, y, x) float32 4MB dask.array<chunksize=(1, 1, 721, 1440), meta=np.ndarray>
    CAPE                                 (valid_time, surface, y, x) float32 4MB dask.array<chunksize=(1, 1, 721, 1440), meta=np.ndarray>
    CICEP                                (valid_time, surface, y, x) float32 4MB dask.array<chunksize=(1, 1, 721, 1440), meta=np.ndarray>
    CIN_1                                (valid_time, pressure_difference_from_ground_2, y, x) float32 12MB dask.array<chunksize=(1, 1, 721, 1440), meta=np.ndarray>
    CAPE_1                               (valid_time, pressure_difference_from_ground_2, y, x) float32 12MB dask.array<chunksize=(1, 1, 721, 1440), meta=np.ndarray>
    ...                                   ...
    VWSH                                 (valid_time, tropopause, y, x) float32 4MB dask.array<chunksize=(1, 1, 721, 1440), meta=np.ndarray>
    VWSH_1                               (valid_time, potential_vorticity_surface, y, x) float32 8MB dask.array<chunksize=(1, 1, 721, 1440), meta=np.ndarray>
    WEASD                                (valid_time, surface, y, x) float32 4MB dask.array<chunksize=(1, 1, 721, 1440), meta=np.ndarray>
    WILT                                 (valid_time, surface, y, x) float32 4MB dask.array<chunksize=(1, 1, 721, 1440), meta=np.ndarray>
    VVEL_1                               (valid_time, sigma_level_2, y, x) float32 4MB dask.array<chunksize=(1, 1, 721, 1440), meta=np.ndarray>
    VVEL                                 (valid_time, isobaric_surface, y, x) float32 170MB dask.array<chunksize=(1, 1, 721, 1440), meta=np.ndarray>

In [6]:
vds

<xarray.Dataset> Size: 3GB
Dimensions:                              (valid_time: 1, mean_sea_level: 1,
                                          y: 721, x: 1440, hybrid_level: 1,
                                          hybrid_level_2: 2,
                                          entire_atmosphere: 1, surface: 1,
                                          planetary_boundary_layer: 1,
                                          isobaric_surface: 41,
                                          ...
                                          highest_tropospheric_freezing_level: 1,
                                          pressure_difference_from_ground: 1,
                                          pressure_difference_from_ground_2: 3,
                                          sigma_level: 4, sigma_level_2: 1,
                                          pressure_difference_from_ground_3: 1,
                                          potential_vorticity_surface: 2)
Coordinates: (12/35)
    valid_time                           (valid_time) int64 8B ManifestArray<...
    mean_sea_level                       (mean_sea_level) float64 8B Manifest...
    hybrid_level                         (hybrid_level) float64 8B ManifestAr...
    hybrid_level_2                       (hybrid_level_2) float64 16B Manifes...
    entire_atmosphere                    (entire_atmosphere) float64 8B Manif...
    surface                              (surface) float64 8B ManifestArray<s...
    ...                                   ...
    pressure_difference_from_ground      (pressure_difference_from_ground) float64 8B ManifestArray<shape=(1,), dtype=float64, chu...
    pressure_difference_from_ground_2    (pressure_difference_from_ground_2) float64 24B ManifestArray<shape=(3,), dtype=float64, ...
    sigma_level                          (sigma_level) float64 32B ManifestAr...
    sigma_level_2                        (sigma_level_2) float64 8B ManifestA...
    pressure_difference_from_ground_3    (pressure_difference_from_ground_3) float64 8B ManifestArray<shape=(1,), dtype=float64, c...
    potential_vorticity_surface          (potential_vorticity_surface) float64 16B ManifestArray<shape=(2,), dtype=float64, chunks...
Dimensions without coordinates: y, x
Data variables: (12/125)
    PRMSL                                (valid_time, mean_sea_level, y, x) float32 4MB ManifestArray<shape=(1, 1, 721, 1440), dtype=float32, chunks=(1, 1, ...
    CLMR                                 (valid_time, hybrid_level, y, x) float32 4MB ManifestArray<shape=(1, 1, 721, 1440), dtype=float32, chunks=(1, 1, 72...
    ICMR                                 (valid_time, hybrid_level, y, x) float32 4MB ManifestArray<shape=(1, 1, 721, 1440), dtype=float32, chunks=(1, 1, 72...
    RWMR                                 (valid_time, hybrid_level, y, x) float32 4MB ManifestArray<shape=(1, 1, 721, 1440), dtype=float32, chunks=(1, 1, 72...
    SNMR                                 (valid_time, hybrid_level, y, x) float32 4MB ManifestArray<shape=(1, 1, 721, 1440), dtype=float32, chunks=(1, 1, 72...
    GRLE                                 (valid_time, hybrid_level, y, x) float32 4MB ManifestArray<shape=(1, 1, 721, 1440), dtype=float32, chunks=(1, 1, 72...
    ...                                   ...
    UGRD_8                               (valid_time, potential_vorticity_surface, y, x) float32 8MB ManifestArray<shape=(1, 2, 721, 1440), dtype=float32, c...
    VGRD_8                               (valid_time, potential_vorticity_surface, y, x) float32 8MB ManifestArray<shape=(1, 2, 721, 1440), dtype=float32, c...
    TMP_8                                (valid_time, potential_vorticity_surface, y, x) float32 8MB ManifestArray<shape=(1, 2, 721, 1440), dtype=float32, c...
    HGT_7                                (valid_time, potential_vorticity_surface, y, x) float32 8MB ManifestArray<shape=(1, 2, 721, 1440), dtype=float32, c...
    PRES_4                               (valid_time, potential_vorticity_

In [ ]:
if "TMP" in ds.data_vars:
    data = ds.TMP.isel(valid_time=0, isobaric_surface=0, y=slice(100, 200), x=slice(100, 200)).compute()
    print(f"shape: {data.shape}, min: {float(data.min()):.2f} K, max: {float(data.max()):.2f} K")
    display(data)

shape: (100, 100), min: 171.92 K, max: 185.92 K


<xarray.DataArray 'TMP' (y: 100, x: 100)> Size: 40kB
array([[177.80804, 177.71805, 177.63805, ..., 171.93805, 171.92804,
        171.93805],
       [177.88805, 177.80804, 177.71805, ..., 171.95804, 171.93805,
        171.91805],
       [177.97804, 177.89804, 177.79805, ..., 171.98804, 171.94804,
        171.91805],
       ...,
       [183.69804, 183.64804, 183.61804, ..., 185.63805, 185.63805,
        185.61804],
       [183.86804, 183.83804, 183.81804, ..., 185.74805, 185.72804,
        185.69804],
       [184.00804, 184.01804, 184.02805, ..., 185.80804, 185.78804,
        185.74805]], shape=(100, 100), dtype=float32)
Coordinates:
    isobaric_surface  float64 8B 1.0
    valid_time        datetime64[ns] 8B 2024-05-01
Dimensions without coordinates: y, x
Attributes:
    _ARRAY_DIMENSIONS:         ['valid_time', 'isobaric_surface', 'y', 'x']
    discipline:                0
    parameterCategory:         0
    parameterNumber:           0
    typeOfFirstFixedSurface:   100
    valueOfFirstFixedSurface:  1.0
    valid_time:                2024-05-01T00:00:00
    shortName:                 TMP
    fullName:                  Temperature
    units:                     K